# 03A - Emotion2Vec/Pretrained Raw-Audio Backbone Fine-tuning 5-Fold + 10-Fold

Notebook này train pretrained speech backbone từ raw audio theo đúng protocol speaker-independent của IEMOCAP.

Mục tiêu:

- Chạy cả **5-fold theo session** và **10-fold theo speaker**.
- Chỉ fine-tune trên `train`, chọn checkpoint bằng `validation`, báo cáo trên `test`.
- Tự dùng **2 GPU T4x2** bằng `torch.nn.DataParallel` nếu Kaggle cấp 2 GPU.
- Lưu checkpoint, prediction, embedding/fusion feature theo từng fold.
- Đóng gói toàn bộ output thành file `.zip` sau khi chạy xong.

Ghi chú: nếu có checkpoint Emotion2Vec tương thích HuggingFace, set `PRETRAINED_MODEL_NAME` sang checkpoint đó. Mặc định notebook dùng `facebook/wav2vec2-base` để tải ổn hơn trên Kaggle; có thể đổi sang `microsoft/wavlm-base-plus` nếu cache/download ổn định.

## 03A update: lightweight pretrained speech branch

Bản update này biến 03A thành nhánh pretrained speech **nhẹ hơn** để có thể chạy trước và tái sử dụng trong fusion sau này.

Thiết lập mặc định:

- `RUN_PROTOCOLS=5fold_session`: chạy 5-fold trước, chưa chạy 10-fold.
- `UNFREEZE_LAST_N=0`: freeze toàn bộ pretrained backbone, chỉ train pooling/adapter/head.
- `EVAL_TRAIN_SPLIT=1`: xuất đủ train/val/test features theo từng fold.
- Output fusion vẫn là `*_pretrained_features.npz`, có thể ghép với 03B/04 sau này.

Ý nghĩa: 03A không còn là full fine-tune nặng. Nó là một pretrained speech feature branch fold-safe, giúp kiểm tra raw-audio pretrained signal và làm nguồn embedding cho fusion.


## Kaggle-safe model default

Notebook 03A trước đó mặc định dùng `microsoft/wavlm-base-plus`, nhưng Kaggle có thể bị kẹt ở bước tải `pytorch_model.bin`.

Bản này đổi mặc định sang:

```text
PRETRAINED_MODEL_NAME=facebook/wav2vec2-base
```

`wav2vec2-base` vẫn là pretrained raw-audio backbone, nhẹ và thường tải ổn hơn trên Kaggle. Nếu muốn thử lại WavLM, set:

```text
PRETRAINED_MODEL_NAME=microsoft/wavlm-base-plus
```


<!-- AUTO-FULL-RUN-GUARD -->
## Chế độ chạy full

Notebook này hiện mặc định chạy full:

```text
RUN_MODE=full
RUN_PROTOCOLS=5fold_session,10fold_speaker
MAX_FOLDS=0
EVAL_TRAIN_SPLIT=1
```

Nếu muốn notebook dừng ngay khi cấu hình chưa phải full, set:

```text
REQUIRE_FULL_RUN=1
```

03A cần raw audio IEMOCAP full release vì pretrained speech backbone nhận waveform trực tiếp.


<!-- AUTO-03ABC-REFERENCE -->
## Nhánh 03A đang kiểm chứng điều gì?

03A là nhánh **pretrained speech / raw-audio backbone**. Khác với 03B, nhánh này không đọc MFCC/log-Mel đã trích sẵn, mà đưa waveform vào một backbone tự giám sát đã học từ lượng lớn audio.

Mục tiêu của nhánh này:

| Câu hỏi | Cách notebook trả lời |
|---|---|
| Pretrained speech model có tốt hơn acoustic thủ công không? | Train 5-fold và 10-fold trên cùng split với 03B/03C |
| Có học được cả emotion class và VAD không? | Dùng 2 head: classification + regression |
| Có thể đưa sang fusion 04 không? | Xuất embedding, emotion probability và VAD prediction cho từng fold |
| Có tránh leakage không? | Mỗi fold train model riêng, test chỉ được dùng sau khi chọn best epoch bằng validation |

### Paper và mô hình tham khảo cho 03A

| Paper / model | Input | Split / task | Kiến trúc sơ bộ | Kết quả tham khảo | Ý nghĩa với 03A |
|---|---|---|---|---|---|
| [Chen & Rudnicky, 2021 - Exploring Wav2Vec2 fine-tuning for SER](https://arxiv.org/abs/2110.06309) | Raw waveform | IEMOCAP 4-class SER, leave-one-session style | wav2vec2 backbone, fine-tuning, TAPT/P-TAPT | P-TAPT báo cáo UA khoảng 74.3 trên IEMOCAP | Cho thấy fine-tune pretrained speech backbone là baseline mạnh hơn handcrafted-only |
| [emotion2vec, 2023](https://arxiv.org/abs/2312.15185) | Raw speech / speech representation | SER downstream | self-supervised emotion representation + downstream classifier | Paper nhấn mạnh representation học cảm xúc dùng được cho nhiều SER benchmarks | Lý do ta muốn có nhánh Emotion2Vec/pretrained speech riêng |
| [Ispas et al., 2024 - Multi-task multi-modal categorical + dimensional emotions](https://arxiv.org/abs/2401.00536) | Audio + transcript | IEMOCAP 10-fold speaker guideline | HuBERT-large + DeBERTaV3-large, self-attention, bridge tokens, multi-task heads | UAR 74.68, WAR 74.69, Valence CCC 0.738, Arousal CCC 0.685 | Cho thấy multi-task categorical + dimensional emotion là hướng hợp lý |

### Kiến trúc 03A

```mermaid
flowchart LR
    A["Raw waveform 16 kHz"] --> B["Pretrained speech backbone<br/>WavLM / wav2vec2 / Emotion2Vec-compatible"]
    B --> C["Frame hidden states"]
    C --> D["Attentive statistics pooling<br/>mean + std có trọng số"]
    D --> E["Shared projection"]
    E --> F["Emotion head<br/>4-class softmax"]
    E --> G["VAD regression head<br/>valence/arousal/dominance"]
    E --> H["Fusion embedding for notebook 04"]
```

### Vì sao dùng attentive statistics pooling?

Mean pooling thường làm mất đoạn cảm xúc ngắn nhưng quan trọng. Attentive statistics pooling học trọng số theo frame, rồi lấy cả trung bình và độ lệch chuẩn có trọng số. Vì vậy embedding giữ được cả mức cảm xúc trung bình và mức biến thiên theo thời gian.

## Dữ liệu cần upload

Notebook cần raw audio, không chỉ feature cache:

```text
data/
  audio_wav/*.wav
  splits/iemocap_5fold_session_long.csv
  splits/iemocap_10fold_speaker_long.csv
  metadata/iemocap_metadata_full.csv    # tùy chọn
```

Output quan trọng cho bước 04:

```text
output_03a_pretrained_backbone/
  models/
  reports/
  fusion_features/
```

`fusion_features` chứa embedding/prediction của train/val/test theo từng fold để ghép với co-attention model mà không làm rò rỉ test.

## Quy trình train-only theo từng fold

Với mỗi fold, notebook làm đúng ba bước:

1. Train backbone/head bằng các mẫu `train` của fold đó.
2. Đánh giá `validation` sau mỗi epoch để chọn checkpoint tốt nhất.
3. Load lại checkpoint tốt nhất rồi mới chạy `train`, `val`, `test` để xuất prediction và fusion feature.

Điểm cần chú ý là `test` không tham gia optimizer, không dùng để chọn epoch, và không dùng để chỉnh tham số. Nhờ vậy output của 03A có thể dùng tiếp trong notebook 04 mà vẫn giữ chuẩn speaker-independent.

## Cách chạy khuyến nghị sau khi thấy run cũ quá chậm/yếu

Run cũ đang fine-tune last 4 layer trong 8 epoch và chạy nhiều fold ngay từ đầu. Với raw-audio backbone, cách đó rất tốn thời gian nhưng chưa đủ kiểm soát để biết model đang yếu vì hyperparameter, pooling, split hay do backbone.

Notebook này được chỉnh lại theo hai giai đoạn:

### Giai đoạn A - tune nhanh nhưng học kỹ hơn

Mặc định:

```text
RUN_MODE=tune
RUN_PROTOCOLS=5fold_session
MAX_FOLDS=1
EPOCHS=20
PATIENCE=6
MAX_SECONDS=4.0
```

Mục tiêu là xem 1 fold có vượt rõ baseline cũ không trước khi đốt GPU cho toàn bộ 5-fold/10-fold.

### Giai đoạn B - chạy full protocol

Khi cấu hình đã ổn, set:

```text
RUN_MODE=full
RUN_PROTOCOLS=5fold_session,10fold_speaker
MAX_FOLDS=0
EPOCHS=12
MAX_SECONDS=6.0
EVAL_TRAIN_SPLIT=1
```

Output `fusion_features` của train/val/test theo từng fold dùng để đưa sang notebook 04.

Các thay đổi so với bản cũ:

- Attention statistics pooling thay cho mean pooling.
- Class weights + label smoothing cho emotion classification.
- Warmup + cosine scheduler theo step.
- Early stopping thật sự bằng `PATIENCE`.
- DataLoader có `NUM_WORKERS` để giảm nghẽn khi đọc/resample wav.


In [ ]:
import os
import sys
import time
import json
import math
import random
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
except Exception:
    plt = None
    sns = None

warnings.filterwarnings(
    "ignore",
    message="Support for mismatched key_padding_mask and attn_mask is deprecated.*",
    category=UserWarning,
)

SEED = int(os.getenv("SEED", "42"))

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
USE_DATA_PARALLEL = os.getenv("USE_DATA_PARALLEL", "1") == "1" and N_GPUS > 1
USE_AMP = DEVICE.type == "cuda"

EMOTION_ID_TO_NAME = {0: "neutral", 1: "angry", 2: "sad", 3: "happy"}
NUM_CLASSES = 4

print("Python:", sys.version)
print("Device:", DEVICE)
print("GPU count:", N_GPUS)
print("Use DataParallel:", USE_DATA_PARALLEL)

In [ ]:
INSTALL_DEPS = os.getenv("INSTALL_DEPS", "1") == "1"
if INSTALL_DEPS:
    import subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "-U",
        "transformers==4.51.3",
        "tokenizers==0.21.1",
        "huggingface_hub==0.30.2",
        "accelerate",
        "safetensors",
        "soundfile",
        "librosa",
        "scikit-learn",
    ])

try:
    import soundfile as sf
except Exception as exc:
    raise ImportError("Thiếu soundfile. Trên Kaggle hãy bật Internet và set INSTALL_DEPS=1.") from exc

try:
    from transformers import AutoFeatureExtractor, AutoModel
except Exception as exc:
    raise ImportError("Thiếu transformers. Trên Kaggle hãy bật Internet và set INSTALL_DEPS=1.") from exc

In [ ]:
LOCAL_PROJECT = Path(r"D:\UTE\Speech Programming\Speech Project")

def unique_existing(paths):
    out = []
    seen = set()
    for item in paths:
        if not item:
            continue
        path = Path(item)
        if path.exists():
            key = str(path.resolve()).lower()
            if key not in seen:
                seen.add(key)
                out.append(path.resolve())
    return out

def search_roots():
    roots = [
        os.getenv("IEMOCAP_DATA_DIR"),
        os.getenv("KAGGLE_INPUT_DIR"),
        Path.cwd(),
        Path.cwd().parent,
        LOCAL_PROJECT / "06_w2v_based_models" / "data",
        LOCAL_PROJECT / "06_w2v_based_models",
        "/kaggle/input",
        "/kaggle/working",
    ]
    return unique_existing(roots)

def find_named_file(filename, env_var=None):
    if env_var and os.getenv(env_var):
        candidate = Path(os.getenv(env_var))
        if candidate.exists():
            return candidate.resolve()
    candidates = []
    for root in search_roots():
        candidates.extend([
            root / filename,
            root / "data" / filename,
            root / "splits" / filename,
            root / "features" / filename,
            root / "metadata" / filename,
            root / "output" / filename,
            root / "output" / "splits" / filename,
            root / "output" / "features" / filename,
            root / "output" / "metadata" / filename,
        ])
        try:
            candidates.extend(root.rglob(filename))
        except Exception:
            pass
    existing = [p.resolve() for p in candidates if p.exists() and p.is_file()]
    if existing:
        return sorted(set(existing), key=lambda p: (len(p.parts), str(p).lower()))[0]
    roots_text = "\n".join(f"- {root}" for root in search_roots())
    raise FileNotFoundError(f"Không tìm thấy {filename}. Notebook đã quét:\n{roots_text}")

def looks_like_iemocap_release(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    sessions = [path / f"Session{i}" for i in range(1, 6)]
    has_sessions = sum(s.exists() for s in sessions) >= 3
    has_wav = any((s / "sentences" / "wav").exists() for s in sessions)
    return has_sessions and has_wav

def has_any_wav(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    try:
        return any(path.rglob("*.wav"))
    except Exception:
        return False

def find_audio_dir():
    """Trả về thư mục có thể quét được wav.

    Hỗ trợ cả:
    - data/audio_wav/*.wav hoặc output/audio_wav/*.wav
    - IEMOCAP_full_release/Session*/sentences/wav/**/*.wav
    """
    if os.getenv("IEMOCAP_AUDIO_DIR"):
        candidate = Path(os.getenv("IEMOCAP_AUDIO_DIR"))
        if candidate.exists() and (has_any_wav(candidate) or looks_like_iemocap_release(candidate)):
            return candidate.resolve()
    if os.getenv("IEMOCAP_ROOT"):
        candidate = Path(os.getenv("IEMOCAP_ROOT"))
        if candidate.exists() and (has_any_wav(candidate) or looks_like_iemocap_release(candidate)):
            return candidate.resolve()
    for root in search_roots():
        for candidate in [
            root / "audio_wav",
            root / "data" / "audio_wav",
            root / "output" / "audio_wav",
            root / "datasets" / "AbstractTTS_IEMOCAP" / "audio_wav",
            root / "IEMOCAP_full_release",
            root / "iemocapfullrelease" / "IEMOCAP_full_release",
            root / "data" / "IEMOCAP_full_release",
        ]:
            if candidate.exists() and (has_any_wav(candidate) or looks_like_iemocap_release(candidate)):
                return candidate.resolve()
        try:
            for candidate in root.rglob("audio_wav"):
                if candidate.is_dir() and has_any_wav(candidate):
                    return candidate.resolve()
            for candidate in root.rglob("IEMOCAP_full_release"):
                if looks_like_iemocap_release(candidate):
                    return candidate.resolve()
            for session1 in root.rglob("Session1"):
                candidate = session1.parent
                if looks_like_iemocap_release(candidate):
                    return candidate.resolve()
        except Exception:
            pass
    roots_text = "\n".join(f"- {root}" for root in search_roots())
    raise FileNotFoundError(
        "Không tìm thấy raw audio IEMOCAP. Notebook đã quét cả dạng audio_wav và "
        f"IEMOCAP_full_release/Session*/sentences/wav:\n{roots_text}"
    )

SPLIT_5FOLD_PATH = find_named_file("iemocap_5fold_session_long.csv", env_var="IEMOCAP_5FOLD_SPLIT_PATH")
SPLIT_10FOLD_PATH = find_named_file("iemocap_10fold_speaker_long.csv", env_var="IEMOCAP_10FOLD_SPLIT_PATH")

def fold_sort_key(name):
    import re
    match = re.search(r"fold_(\d+)", str(name))
    return int(match.group(1)) if match else str(name)

def load_split_table(path, protocol):
    df = pd.read_csv(path)
    required = {"utterance_id", "speaker_id", "session", "emotion_4class", "emotion_id", "valence", "arousal", "dominance", "wav_path", "fold", "split", "train_sample_id"}
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Thiếu cột trong split {protocol}: {missing}")
    df = df.copy()
    df["protocol"] = protocol
    split_map = {
        "train": "train",
        "training": "train",
        "val": "val",
        "valid": "val",
        "validation": "val",
        "dev": "val",
        "test": "test",
        "testing": "test",
    }
    original_split_values = sorted(df["split"].astype(str).str.lower().str.strip().unique().tolist())
    df["split_original"] = df["split"].astype(str)
    df["split"] = df["split"].astype(str).str.lower().str.strip().map(split_map).fillna(df["split"].astype(str).str.lower().str.strip())
    normalized_split_values = sorted(df["split"].unique().tolist())
    print(f"{protocol} split labels:", original_split_values, "->", normalized_split_values)
    df["emotion_id"] = df["emotion_id"].astype(int)
    for col in ["valence", "arousal", "dominance"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df.dropna(subset=["valence", "arousal", "dominance"]).reset_index(drop=True)

def assert_fold_has_required_splits(protocol, fold_name, fold_df):
    counts = fold_df["split"].value_counts().to_dict()
    missing = [name for name in ["train", "val", "test"] if counts.get(name, 0) == 0]
    if missing:
        raise ValueError(
            f"Fold {protocol}/{fold_name} thiếu split {missing}. "
            f"Số lượng hiện có: {counts}. Hãy kiểm tra cột split trong file CSV."
        )

SPLIT_TABLES = {
    "5fold_session": load_split_table(SPLIT_5FOLD_PATH, "5fold_session"),
    "10fold_speaker": load_split_table(SPLIT_10FOLD_PATH, "10fold_speaker"),
}

for protocol, df in SPLIT_TABLES.items():
    print(protocol, "rows:", len(df), "folds:", df["fold"].nunique())
    display(df.groupby(["fold", "split"]).size().unstack(fill_value=0).head(20))

In [ ]:
AUDIO_DIR = find_audio_dir()
print("AUDIO_DIR:", AUDIO_DIR)
print("SPLIT_5FOLD_PATH:", SPLIT_5FOLD_PATH)
print("SPLIT_10FOLD_PATH:", SPLIT_10FOLD_PATH)

RUN_MODE = os.getenv("RUN_MODE", "full").strip().lower()
IS_TUNE_MODE = RUN_MODE != "full"

PRETRAINED_MODEL_NAME = os.getenv("PRETRAINED_MODEL_NAME", "facebook/wav2vec2-base")
PRETRAINED_MODEL_PATH = os.getenv("PRETRAINED_MODEL_PATH", "").strip()
ALLOW_HF_DOWNLOAD = os.getenv("ALLOW_HF_DOWNLOAD", "0") == "1"
SAMPLE_RATE = int(os.getenv("SAMPLE_RATE", "16000"))
MAX_SECONDS = float(os.getenv("MAX_SECONDS", "4.0" if IS_TUNE_MODE else "6.0"))
MAX_SAMPLES = int(SAMPLE_RATE * MAX_SECONDS)

EPOCHS = int(os.getenv("EPOCHS", "20" if IS_TUNE_MODE else "20"))
PATIENCE = int(os.getenv("PATIENCE", "6" if IS_TUNE_MODE else "6"))
MIN_DELTA = float(os.getenv("MIN_DELTA", "0.002"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "8"))
GRAD_ACCUM = int(os.getenv("GRAD_ACCUM", "2"))
NUM_WORKERS = int(os.getenv("NUM_WORKERS", "2"))

LR_BACKBONE = float(os.getenv("LR_BACKBONE", "0.0"))
LR_HEAD = float(os.getenv("LR_HEAD", "5e-4"))
MIN_LR_RATIO = float(os.getenv("MIN_LR_RATIO", "0.05"))
WARMUP_RATIO = float(os.getenv("WARMUP_RATIO", "0.08"))
WEIGHT_DECAY = float(os.getenv("WEIGHT_DECAY", "1e-4"))
DROPOUT = float(os.getenv("DROPOUT", "0.30"))
HIDDEN_DIM = int(os.getenv("HIDDEN_DIM", "256"))
UNFREEZE_LAST_N = int(os.getenv("UNFREEZE_LAST_N", "0"))
LABEL_SMOOTHING = float(os.getenv("LABEL_SMOOTHING", "0.05"))
USE_CLASS_WEIGHTS = os.getenv("USE_CLASS_WEIGHTS", "1") == "1"
USE_SCHEDULER = os.getenv("USE_SCHEDULER", "1") == "1"

MAX_FOLDS = int(os.getenv("MAX_FOLDS", "1" if IS_TUNE_MODE else "0"))
RUN_PROTOCOLS = [x.strip() for x in os.getenv("RUN_PROTOCOLS", "5fold_session").split(",") if x.strip()]
EVAL_TRAIN_SPLIT = os.getenv("EVAL_TRAIN_SPLIT", "1") == "1"


REQUIRE_FULL_RUN = os.getenv("REQUIRE_FULL_RUN", "0") == "1"
if REQUIRE_FULL_RUN:
    expected_protocols = {"5fold_session", "10fold_speaker"}
    if RUN_MODE != "full" or MAX_FOLDS != 0 or set(RUN_PROTOCOLS) != expected_protocols or not EVAL_TRAIN_SPLIT:
        raise ValueError(
            "REQUIRE_FULL_RUN=1 nhưng cấu hình chưa phải full. "
            f"RUN_MODE={RUN_MODE}, MAX_FOLDS={MAX_FOLDS}, RUN_PROTOCOLS={RUN_PROTOCOLS}, "
            f"EVAL_TRAIN_SPLIT={EVAL_TRAIN_SPLIT}. "
            "Cần RUN_MODE=full, MAX_FOLDS=0, RUN_PROTOCOLS=5fold_session,10fold_speaker, EVAL_TRAIN_SPLIT=1."
        )

OUTPUT_DIR = Path(os.getenv("OUTPUT_DIR", "output_03a_pretrained_backbone")).resolve()
MODEL_DIR = OUTPUT_DIR / "models"
REPORT_DIR = OUTPUT_DIR / "reports"
FUSION_DIR = OUTPUT_DIR / "fusion_features"
FIGURE_DIR = OUTPUT_DIR / "figures"
for p in [MODEL_DIR, REPORT_DIR, FUSION_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

LIVE_LOG_PATH = REPORT_DIR / "03A_live_training_log.txt"
LOG_EVERY_STEPS = int(os.getenv("LOG_EVERY_STEPS", "25"))

def live_log(message):
    stamp = time.strftime("%Y-%m-%d %H:%M:%S")
    text = f"[{stamp}] {message}"
    print(text, flush=True)
    with LIVE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(text + "\n")

print("Run mode:", RUN_MODE)
print("Pretrained model:", PRETRAINED_MODEL_NAME)
print("Pretrained model path:", PRETRAINED_MODEL_PATH)
print("Allow HF download:", ALLOW_HF_DOWNLOAD)
print("Output:", OUTPUT_DIR)
print({
    "MAX_SECONDS": MAX_SECONDS,
    "EPOCHS": EPOCHS,
    "PATIENCE": PATIENCE,
    "BATCH_SIZE": BATCH_SIZE,
    "GRAD_ACCUM": GRAD_ACCUM,
    "NUM_WORKERS": NUM_WORKERS,
    "LR_BACKBONE": LR_BACKBONE,
    "LR_HEAD": LR_HEAD,
    "UNFREEZE_LAST_N": UNFREEZE_LAST_N,
    "LABEL_SMOOTHING": LABEL_SMOOTHING,
    "RUN_PROTOCOLS": RUN_PROTOCOLS,
    "MAX_FOLDS": MAX_FOLDS,
    "EVAL_TRAIN_SPLIT": EVAL_TRAIN_SPLIT,
    "LOG_EVERY_STEPS": LOG_EVERY_STEPS,
    "PRETRAINED_MODEL_PATH": PRETRAINED_MODEL_PATH,
    "ALLOW_HF_DOWNLOAD": ALLOW_HF_DOWNLOAD,
})
def maybe_extract_model_zips_03a():
    extract_dir = Path("extracted_03a_models")
    extract_dir.mkdir(parents=True, exist_ok=True)
    for root in search_roots():
        try:
            zips = list(root.rglob("*.zip"))
        except Exception:
            continue
        for z in zips:
            lname = z.name.lower()
            if not any(k in lname for k in ["wav2vec", "wavlm", "hubert", "speech", "pretrained", "huggingface", "hf-model"]):
                continue
            target = extract_dir / z.stem
            marker = target / ".extracted"
            if marker.exists():
                continue
            try:
                target.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(z, "r") as zipf:
                    zipf.extractall(target)
                marker.write_text("ok", encoding="utf-8")
                live_log(f"Extracted local speech model zip: {z}")
            except Exception as exc:
                live_log(f"Không giải nén được speech model zip {z}: {exc}")

def _norm_model_name(value):
    return str(value).lower().replace("\\", "/").replace("_", "-").replace(".", "-")

def _looks_like_hf_speech_model(path):
    path = Path(path)
    if not path.is_dir():
        return False
    has_config = (path / "config.json").exists()
    has_weight = any((path / name).exists() for name in ["model.safetensors", "pytorch_model.bin"])
    has_feature = any((path / name).exists() for name in ["preprocessor_config.json", "feature_extractor_config.json"])
    return has_config and has_weight and has_feature

def find_local_speech_model(model_name):
    maybe_extract_model_zips_03a()
    if PRETRAINED_MODEL_PATH:
        p = Path(PRETRAINED_MODEL_PATH)
        if _looks_like_hf_speech_model(p):
            return p.resolve()
        raise FileNotFoundError(
            f"PRETRAINED_MODEL_PATH không hợp lệ: {p}. Cần folder có config.json, weight file và preprocessor_config.json."
        )
    wanted = _norm_model_name(model_name).split("/")[-1]
    candidates = []
    for root in search_roots() + [Path("extracted_03a_models")]:
        try:
            for cfg in root.rglob("config.json"):
                folder = cfg.parent
                if _looks_like_hf_speech_model(folder):
                    score = 0 if wanted in _norm_model_name(folder) else 1
                    candidates.append((score, len(folder.parts), str(folder).lower(), folder.resolve()))
        except Exception:
            continue
    if candidates:
        return sorted(candidates)[0][-1]
    return None

PRETRAINED_MODEL_SOURCE = find_local_speech_model(PRETRAINED_MODEL_NAME)
if PRETRAINED_MODEL_SOURCE is None:
    if not ALLOW_HF_DOWNLOAD:
        roots = "\n".join(f"- {r}" for r in search_roots())
        raise FileNotFoundError(
            "03A không tìm thấy local pretrained speech model. Notebook chặn tải online mặc định để tránh treo.\n"
            "Hãy upload wav2vec2-base-kaggle.zip vào Kaggle Input, hoặc set PRETRAINED_MODEL_PATH tới folder model.\n"
            "Nếu vẫn muốn tải online, set ALLOW_HF_DOWNLOAD=1.\n\n"
            f"Đã quét:\n{roots}"
        )
    PRETRAINED_MODEL_SOURCE = PRETRAINED_MODEL_NAME
else:
    PRETRAINED_MODEL_SOURCE = str(PRETRAINED_MODEL_SOURCE)

live_log("03A configuration is ready.")
live_log(f"PRETRAINED_MODEL_SOURCE={PRETRAINED_MODEL_SOURCE}")
live_log(f"ALLOW_HF_DOWNLOAD={ALLOW_HF_DOWNLOAD}")


In [ ]:
def vad_to_0_1(values):
    return np.clip((values.astype(np.float32) - 1.0) / 4.0, 0.0, 1.0)

def vad_from_0_1(values):
    return values.astype(np.float32) * 4.0 + 1.0

def concordance_ccc_torch(pred, target, eps=1e-8):
    pred_mean = pred.mean(dim=0)
    target_mean = target.mean(dim=0)
    pred_var = pred.var(dim=0, unbiased=False)
    target_var = target.var(dim=0, unbiased=False)
    cov = ((pred - pred_mean) * (target - target_mean)).mean(dim=0)
    return (2.0 * cov) / (pred_var + target_var + (pred_mean - target_mean).pow(2) + eps)

def concordance_ccc_np(pred, true, eps=1e-8):
    pred = np.asarray(pred, dtype=np.float64)
    true = np.asarray(true, dtype=np.float64)
    pred_mean = pred.mean(axis=0)
    true_mean = true.mean(axis=0)
    pred_var = pred.var(axis=0)
    true_var = true.var(axis=0)
    cov = ((pred - pred_mean) * (true - true_mean)).mean(axis=0)
    return (2.0 * cov) / (pred_var + true_var + (pred_mean - true_mean) ** 2 + eps)

def compute_metrics(y_true, y_pred, vad_true_01, vad_pred_01):
    vad_true_raw = vad_from_0_1(np.asarray(vad_true_01))
    vad_pred_raw = vad_from_0_1(np.asarray(vad_pred_01))
    ccc = concordance_ccc_np(vad_pred_raw, vad_true_raw)
    mae = np.abs(vad_pred_raw - vad_true_raw).mean(axis=0)
    rmse = np.sqrt(((vad_pred_raw - vad_true_raw) ** 2).mean(axis=0))
    return {
        "WA": float(accuracy_score(y_true, y_pred)),
        "UAR": float(balanced_accuracy_score(y_true, y_pred)),
        "Macro_F1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "Weighted_F1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "CCC_valence": float(ccc[0]),
        "CCC_arousal": float(ccc[1]),
        "CCC_dominance": float(ccc[2]),
        "CCC_mean": float(np.mean(ccc)),
        "MAE_mean": float(np.mean(mae)),
        "RMSE_mean": float(np.mean(rmse)),
    }

def primary_score(metrics):
    return 0.40 * metrics["UAR"] + 0.20 * metrics["WA"] + 0.20 * metrics["Macro_F1"] + 0.20 * metrics["CCC_mean"]

def class_weights_for(df):
    labels = df["emotion_id"].astype(int).to_numpy()
    counts = np.bincount(labels, minlength=NUM_CLASSES).astype(np.float32)
    weights = counts.sum() / np.maximum(counts, 1.0)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def multitask_loss(outputs, emotion_true, vad_true, class_weights=None, ce_weight=1.0, mse_weight=0.35, ccc_weight=0.50):
    ce = F.cross_entropy(
        outputs["emotion_logits"],
        emotion_true,
        weight=class_weights,
        label_smoothing=LABEL_SMOOTHING,
    )
    mse = F.mse_loss(outputs["vad_pred"], vad_true)
    ccc_loss = (1.0 - concordance_ccc_torch(outputs["vad_pred"], vad_true)).mean()
    return ce_weight * ce + mse_weight * mse + ccc_weight * ccc_loss

def module_state_dict(model):
    return model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()

def load_module_state_dict(model, state_dict):
    target = model.module if isinstance(model, nn.DataParallel) else model
    target.load_state_dict(state_dict)

def zip_output(output_dir):
    output_dir = Path(output_dir)
    zip_path = output_dir.with_suffix(".zip")
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in output_dir.rglob("*"):
            if path.is_file():
                zf.write(path, path.relative_to(output_dir.parent))
    print("Saved zip:", zip_path)
    return zip_path


In [ ]:
def make_grad_scaler(enabled):
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        try:
            return torch.amp.GradScaler("cuda", enabled=enabled)
        except TypeError:
            return torch.amp.GradScaler(enabled=enabled)
    return torch.cuda.amp.GradScaler(enabled=enabled)

def autocast_context(enabled):
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        return torch.amp.autocast("cuda", enabled=enabled)
    return torch.cuda.amp.autocast(enabled=enabled)

live_log(f"Loading feature extractor from: {PRETRAINED_MODEL_SOURCE}")
FEATURE_EXTRACTOR = AutoFeatureExtractor.from_pretrained(PRETRAINED_MODEL_SOURCE, local_files_only=not ALLOW_HF_DOWNLOAD)
live_log("Feature extractor loaded.")
_AUDIO_INDEX = None

def audio_index():
    global _AUDIO_INDEX
    if _AUDIO_INDEX is None:
        _AUDIO_INDEX = {}
        for p in AUDIO_DIR.rglob("*.wav"):
            resolved = p.resolve()
            _AUDIO_INDEX[p.name] = resolved
            _AUDIO_INDEX[p.stem] = resolved
    return _AUDIO_INDEX

def resolve_wav_path(row):
    name = Path(str(row.get("wav_path", "")).replace("\\", "/")).name
    direct = AUDIO_DIR / name
    if direct.exists():
        return direct.resolve()
    idx = audio_index()
    if name in idx:
        return idx[name]
    stem = Path(name).stem
    if stem in idx:
        return idx[stem]
    utterance = str(row.get("utterance_id", ""))
    direct = AUDIO_DIR / f"{utterance}.wav"
    if direct.exists():
        return direct.resolve()
    if utterance in idx:
        return idx[utterance]
    if f"{utterance}.wav" in idx:
        return idx[f"{utterance}.wav"]
    raise FileNotFoundError(f"Không tìm thấy wav cho {utterance} ({name})")

def load_audio_16k(path):
    wav, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if wav.ndim == 2:
        wav = wav.mean(axis=1)
    if sr != SAMPLE_RATE:
        import librosa
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SAMPLE_RATE)
    wav = np.asarray(wav, dtype=np.float32)
    peak = float(np.max(np.abs(wav))) if wav.size else 0.0
    if peak > 1.0:
        wav = wav / peak
    return wav

In [ ]:
class RawAudioDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True).copy()
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        vad = row[["valence", "arousal", "dominance"]].to_numpy(dtype=np.float32)
        return {
            "utterance_id": str(row["utterance_id"]),
            "train_sample_id": str(row["train_sample_id"]),
            "speaker_id": str(row["speaker_id"]),
            "session": str(row["session"]),
            "split": str(row["split"]),
            "wav_path": str(resolve_wav_path(row)),
            "emotion_id": int(row["emotion_id"]),
            "vad": vad_to_0_1(vad),
        }

def collate_raw(batch):
    arrays = [load_audio_16k(Path(item["wav_path"])) for item in batch]
    encoded = FEATURE_EXTRACTOR(
        arrays,
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SAMPLES,
    )
    out = {
        "input_values": encoded["input_values"],
        "attention_mask": encoded.get("attention_mask"),
        "emotion_id": torch.tensor([x["emotion_id"] for x in batch], dtype=torch.long),
        "vad": torch.tensor(np.stack([x["vad"] for x in batch]), dtype=torch.float32),
        "utterance_id": [x["utterance_id"] for x in batch],
        "train_sample_id": [x["train_sample_id"] for x in batch],
        "speaker_id": [x["speaker_id"] for x in batch],
        "session": [x["session"] for x in batch],
        "split": [x["split"] for x in batch],
    }
    return out

def make_loader(df, shuffle=False):
    kwargs = {
        "batch_size": BATCH_SIZE,
        "shuffle": shuffle,
        "num_workers": NUM_WORKERS,
        "pin_memory": DEVICE.type == "cuda",
        "collate_fn": collate_raw,
    }
    if NUM_WORKERS > 0:
        kwargs["persistent_workers"] = True
        kwargs["prefetch_factor"] = 2
    return DataLoader(RawAudioDataset(df), **kwargs)

def to_device(batch):
    return {k: (v.to(DEVICE, non_blocking=True) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}

def forward_model(model, input_values, attention_mask=None, return_embedding=False):
    # DataParallel can fail on the final small batch when one GPU receives no samples.
    # For that case, fall back to the underlying module on the main device.
    if isinstance(model, nn.DataParallel) and input_values.size(0) < len(model.device_ids):
        return model.module(input_values, attention_mask, return_embedding=return_embedding)
    return model(input_values, attention_mask, return_embedding=return_embedding)


## Kiến trúc

Backbone pretrained nhận waveform và xuất chuỗi hidden states. Notebook dùng masked mean pooling để tạo utterance embedding, sau đó đi qua shared MLP và hai head:

- `emotion_head`: phân loại 4 cảm xúc.
- `vad_head`: hồi quy valence/arousal/dominance.

Khi Kaggle cấp T4x2, `DataParallel` chia batch qua 2 GPU. Vì batch audio dài khá nặng, có thể tăng `BATCH_SIZE` nếu VRAM cho phép.

In [ ]:
class AttentiveStatisticsPooling(nn.Module):
    def __init__(self, dim, hidden=128, dropout=0.1):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def forward(self, hidden, attention_mask=None):
        scores = self.attn(hidden).squeeze(-1)
        if attention_mask is not None:
            mask = attention_mask
            if mask.shape[1] != hidden.shape[1]:
                mask = F.interpolate(mask.float().unsqueeze(1), size=hidden.shape[1], mode="nearest").squeeze(1)
            scores = scores.masked_fill(mask.to(hidden.device) <= 0, -1e4)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        mean = (hidden * weights).sum(dim=1)
        var = ((hidden - mean.unsqueeze(1)).pow(2) * weights).sum(dim=1).clamp(min=1e-6)
        std = torch.sqrt(var)
        return torch.cat([mean, std], dim=-1), weights.squeeze(-1)

class RawBackboneMultiTaskSER(nn.Module):
    def __init__(self, model_name, hidden_dim=256, dropout=0.30):
        super().__init__()
        live_log(f"Loading raw-audio pretrained backbone: {model_name}")
        self.backbone = AutoModel.from_pretrained(model_name, local_files_only=not ALLOW_HF_DOWNLOAD)
        live_log("Raw-audio pretrained backbone loaded.")
        h = int(getattr(self.backbone.config, "hidden_size", 768))
        self.pool = AttentiveStatisticsPooling(h, hidden=max(128, h // 4), dropout=dropout * 0.5)
        self.shared = nn.Sequential(
            nn.LayerNorm(h * 2),
            nn.Dropout(dropout),
            nn.Linear(h * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
        )
        self.emotion_head = nn.Linear(hidden_dim, NUM_CLASSES)
        self.vad_head = nn.Sequential(nn.Linear(hidden_dim, 3), nn.Sigmoid())

    def forward(self, input_values, attention_mask=None, return_embedding=False):
        outputs = self.backbone(input_values=input_values, attention_mask=attention_mask)
        pooled, attn_weights = self.pool(outputs.last_hidden_state, attention_mask)
        embedding = self.shared(pooled)
        out = {"emotion_logits": self.emotion_head(embedding), "vad_pred": self.vad_head(embedding)}
        if return_embedding:
            out["embedding"] = embedding
            out["attention_weights"] = attn_weights
        return out

def freeze_backbone(model, unfreeze_last_n):
    target = model.module if isinstance(model, nn.DataParallel) else model
    if unfreeze_last_n < 0:
        for p in target.backbone.parameters():
            p.requires_grad = True
        return "Full backbone fine-tuning"
    for p in target.backbone.parameters():
        p.requires_grad = False
    layers = getattr(getattr(target.backbone, "encoder", None), "layers", None)
    if layers is not None and unfreeze_last_n > 0:
        for layer in layers[-unfreeze_last_n:]:
            for p in layer.parameters():
                p.requires_grad = True
        return f"Frozen backbone except last {unfreeze_last_n} encoder layers"
    return "Backbone frozen"

def build_model():
    model = RawBackboneMultiTaskSER(PRETRAINED_MODEL_SOURCE, HIDDEN_DIM, DROPOUT).to(DEVICE)
    note = freeze_backbone(model, UNFREEZE_LAST_N)
    if USE_DATA_PARALLEL:
        model = nn.DataParallel(model)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    live_log(note)
    live_log(f"Trainable parameters: {trainable:,}/{total:,} ({trainable/max(total,1):.2%})")
    return model

def build_optimizer(model):
    backbone, heads = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        clean = name.replace("module.", "")
        if clean.startswith("backbone."):
            backbone.append(param)
        else:
            heads.append(param)
    groups = []
    if backbone:
        groups.append({"params": backbone, "lr": LR_BACKBONE})
    if heads:
        groups.append({"params": heads, "lr": LR_HEAD})
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)

def build_scheduler(optimizer, total_update_steps):
    if not USE_SCHEDULER:
        return None
    warmup_steps = max(1, int(total_update_steps * WARMUP_RATIO))
    total_update_steps = max(warmup_steps + 1, total_update_steps)

    def lr_lambda(step):
        if step < warmup_steps:
            return max(1e-4, float(step + 1) / float(warmup_steps))
        progress = (step - warmup_steps) / float(max(1, total_update_steps - warmup_steps))
        cosine = 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
        return MIN_LR_RATIO + (1.0 - MIN_LR_RATIO) * cosine

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


In [ ]:
@torch.no_grad()
def evaluate(model, loader, split_name, class_weights=None):
    if len(loader.dataset) == 0:
        raise ValueError(f"Split `{split_name}` rỗng, không thể evaluate. Hãy kiểm tra mapping train/val/test của fold.")
    model.eval()
    y_true, y_pred, vad_true, vad_pred, embeddings, probs = [], [], [], [], [], []
    rows = []
    total_loss, n_batches = 0.0, 0
    eval_start = time.time()
    live_log(f"Evaluate {split_name}: batches={len(loader)}")
    for batch_idx, batch in enumerate(loader, start=1):
        batch = to_device(batch)
        outputs = forward_model(model, batch["input_values"], batch.get("attention_mask"), return_embedding=True)
        loss = multitask_loss(outputs, batch["emotion_id"], batch["vad"], class_weights=class_weights)
        prob = torch.softmax(outputs["emotion_logits"], dim=-1)
        pred = prob.argmax(dim=-1)
        y_true.extend(batch["emotion_id"].detach().cpu().numpy().tolist())
        y_pred.extend(pred.detach().cpu().numpy().tolist())
        vad_true.append(batch["vad"].detach().cpu().numpy())
        vad_pred.append(outputs["vad_pred"].detach().cpu().numpy())
        embeddings.append(outputs["embedding"].detach().cpu().numpy())
        probs.append(prob.detach().cpu().numpy())
        for i, uid in enumerate(batch["utterance_id"]):
            rows.append({
                "utterance_id": uid,
                "train_sample_id": batch["train_sample_id"][i],
                "speaker_id": batch["speaker_id"][i],
                "session": batch["session"][i],
                "split": split_name,
            })
        total_loss += float(loss.detach().cpu())
        n_batches += 1
        if batch_idx == 1 or batch_idx % LOG_EVERY_STEPS == 0 or batch_idx == len(loader):
            live_log(f"Evaluate {split_name}: batch {batch_idx}/{len(loader)} elapsed={time.time()-eval_start:.1f}s")
    if not vad_true:
        raise ValueError(f"Không có batch nào trong split `{split_name}`. DataLoader đang rỗng.")
    vad_true = np.concatenate(vad_true)
    vad_pred = np.concatenate(vad_pred)
    embeddings = np.concatenate(embeddings)
    probs = np.concatenate(probs)
    metrics = compute_metrics(y_true, y_pred, vad_true, vad_pred)
    metrics["loss"] = total_loss / max(n_batches, 1)
    pred_df = pd.DataFrame(rows)
    pred_df["true_emotion_id"] = y_true
    pred_df["pred_emotion_id"] = y_pred
    for i, name in EMOTION_ID_TO_NAME.items():
        pred_df[f"prob_{name}"] = probs[:, i]
    for j, name in enumerate(["valence", "arousal", "dominance"]):
        pred_df[f"true_{name}"] = vad_from_0_1(vad_true)[:, j]
        pred_df[f"pred_{name}"] = vad_from_0_1(vad_pred)[:, j]
    feature_npz = {
        "utterance_id": pred_df["utterance_id"].to_numpy(),
        "train_sample_id": pred_df["train_sample_id"].to_numpy(),
        "embedding": embeddings.astype(np.float32),
        "emotion_probs": probs.astype(np.float32),
        "vad_pred": vad_pred.astype(np.float32),
        "emotion_true": np.asarray(y_true, dtype=np.int64),
        "vad_true": vad_true.astype(np.float32),
    }
    return metrics, pred_df, feature_npz


In [ ]:
def train_fold(protocol, fold_name, fold_df, seed):
    set_seed(seed)
    assert_fold_has_required_splits(protocol, fold_name, fold_df)
    train_df = fold_df[fold_df["split"] == "train"].reset_index(drop=True)
    val_df = fold_df[fold_df["split"] == "val"].reset_index(drop=True)
    test_df = fold_df[fold_df["split"] == "test"].reset_index(drop=True)
    live_log(f"=== {protocol} | {fold_name} ===")
    live_log(f"Train/Val/Test: {len(train_df)} {len(val_df)} {len(test_df)}")

    live_log("Build raw-audio DataLoaders")
    train_loader = make_loader(train_df, shuffle=True)
    val_loader = make_loader(val_df, shuffle=False)
    test_loader = make_loader(test_df, shuffle=False)
    live_log(f"DataLoaders ready. Train batches={len(train_loader)}, Val batches={len(val_loader)}, Test batches={len(test_loader)}")

    live_log("Build raw-audio pretrained model")
    model = build_model()
    optimizer = build_optimizer(model)
    updates_per_epoch = max(1, math.ceil(len(train_loader) / max(GRAD_ACCUM, 1)))
    scheduler = build_scheduler(optimizer, updates_per_epoch * EPOCHS)
    scaler = make_grad_scaler(USE_AMP)
    class_weights = class_weights_for(train_df) if USE_CLASS_WEIGHTS else None
    if class_weights is not None:
        live_log("Class weights: " + str([round(float(x), 4) for x in class_weights.detach().cpu()]))

    best_score, best_epoch, bad_epochs = -1e9, -1, 0
    best_path = MODEL_DIR / protocol / f"{fold_name}_best.pt"
    best_path.parent.mkdir(parents=True, exist_ok=True)
    history = []
    global_step = 0

    live_log("Optimizer, scheduler, and AMP scaler are ready. Start training loop.")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = 0.0
        start = time.time()
        live_log(f"Epoch {epoch:03d} started. Train batches={len(train_loader)}")
        for step, batch in enumerate(train_loader, start=1):
            batch = to_device(batch)
            with autocast_context(USE_AMP):
                outputs = forward_model(model, batch["input_values"], batch.get("attention_mask"), return_embedding=False)
                loss = multitask_loss(outputs, batch["emotion_id"], batch["vad"], class_weights=class_weights) / GRAD_ACCUM
            scaler.scale(loss).backward()
            if step % GRAD_ACCUM == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                if scheduler is not None:
                    scheduler.step()
                global_step += 1
            running += float(loss.detach().cpu()) * GRAD_ACCUM
            if step == 1 or step % LOG_EVERY_STEPS == 0 or step == len(train_loader):
                live_log(
                    f"Epoch {epoch:03d} step {step}/{len(train_loader)} "
                    f"loss={running/max(step,1):.4f} elapsed={time.time()-start:.1f}s"
                )

        live_log(f"Epoch {epoch:03d} training done. Start validation.")
        val_metrics, _, _ = evaluate(model, val_loader, "val", class_weights=class_weights)
        score = primary_score(val_metrics)
        current_lrs = [group["lr"] for group in optimizer.param_groups]
        row = {
            "protocol": protocol,
            "fold": fold_name,
            "epoch": epoch,
            "train_loss": running / max(len(train_loader), 1),
            "val_primary_score": score,
            "lr_min": min(current_lrs),
            "lr_max": max(current_lrs),
            **{f"val_{k}": v for k, v in val_metrics.items()},
            "seconds": time.time() - start,
        }
        history.append(row)
        pd.DataFrame(history).to_csv(REPORT_DIR / f"{protocol}_{fold_name}_history_live.csv", index=False, encoding="utf-8-sig")
        live_log(
            f"Epoch {epoch:03d} | train_loss={row['train_loss']:.4f} | "
            f"val_UAR={val_metrics['UAR']:.4f} | val_CCC={val_metrics['CCC_mean']:.4f} | "
            f"score={score:.4f} | lr={row['lr_min']:.2e}-{row['lr_max']:.2e}"
        )
        if score > best_score + MIN_DELTA:
            best_score, best_epoch, bad_epochs = score, epoch, 0
            if best_path.exists():
                best_path.unlink()
            torch.save({"model_state_dict": module_state_dict(model), "best_epoch": int(best_epoch), "best_val_score": float(best_score)}, best_path)
            live_log(f"Saved new best checkpoint: epoch={epoch}, score={best_score:.4f}, path={best_path}")
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                live_log(f"Early stopping at epoch {epoch}; best epoch={best_epoch}, best score={best_score:.4f}")
                break

    live_log(f"Load best checkpoint from {best_path}")
    checkpoint = torch.load(best_path, map_location=DEVICE, weights_only=False)
    load_module_state_dict(model, checkpoint["model_state_dict"])
    split_outputs = {}
    split_loaders = [("val", val_loader), ("test", test_loader)]
    if EVAL_TRAIN_SPLIT:
        split_loaders.insert(0, ("train", train_loader))
    for split_name, loader in split_loaders:
        live_log(f"Evaluate/export split={split_name}")
        metrics, pred_df, feature_npz = evaluate(model, loader, split_name, class_weights=class_weights)
        pred_df.to_csv(REPORT_DIR / f"{protocol}_{fold_name}_{split_name}_predictions.csv", index=False, encoding="utf-8-sig")
        np.savez_compressed(FUSION_DIR / f"{protocol}_{fold_name}_{split_name}_pretrained_features.npz", **feature_npz)
        split_outputs[split_name] = metrics
    history_df = pd.DataFrame(history)
    history_df.to_csv(REPORT_DIR / f"{protocol}_{fold_name}_history.csv", index=False, encoding="utf-8-sig")
    result = {
        "protocol": protocol,
        "fold": fold_name,
        "best_epoch": best_epoch,
        "best_val_score": best_score,
        "n_train": len(train_df),
        "n_val": len(val_df),
        "n_test": len(test_df),
        **split_outputs["test"],
    }
    live_log("Test: " + str({k: result[k] for k in ["WA", "UAR", "Macro_F1", "CCC_mean", "MAE_mean"]}))
    return result


In [ ]:
all_results = []
start_all = time.time()
for protocol in RUN_PROTOCOLS:
    df = SPLIT_TABLES[protocol]
    folds = sorted(df["fold"].unique().tolist(), key=fold_sort_key)
    if MAX_FOLDS > 0:
        folds = folds[:MAX_FOLDS]
    for idx, fold in enumerate(folds, start=1):
        result = train_fold(protocol, fold, df[df["fold"] == fold].reset_index(drop=True), SEED + idx)
        all_results.append(result)
        pd.DataFrame(all_results).to_csv(REPORT_DIR / "03A_results_live.csv", index=False, encoding="utf-8-sig")
        live_log(f"Finished fold {fold}. Live results saved.")

results_df = pd.DataFrame(all_results)
results_df.to_csv(REPORT_DIR / "03a_pretrained_backbone_results_by_fold.csv", index=False, encoding="utf-8-sig")
display(results_df)
summary = results_df.groupby("protocol")[["WA", "UAR", "Macro_F1", "Weighted_F1", "CCC_valence", "CCC_arousal", "CCC_dominance", "CCC_mean", "MAE_mean", "RMSE_mean"]].agg(["mean", "std"]).round(4)
summary.to_csv(REPORT_DIR / "03a_pretrained_backbone_summary.csv", encoding="utf-8-sig")
display(summary)
live_log(f"Total seconds: {round(time.time() - start_all, 2)}")

<!-- AUTO-03ABC-PAPER-RESULTS -->
## Bảng kết quả, biểu đồ và manifest output

Sau khi train xong, cell dưới đây gom kết quả thành bảng kiểu paper, tạo biểu đồ training/validation và ghi manifest các file output để dễ upload sang notebook 04.

In [ ]:
# AUTO-03ABC-PAPER-RESULTS
def _format_mean_std(df, metric_cols):
    rows = []
    for protocol, group in df.groupby("protocol"):
        row = {"Protocol": protocol, "Folds": int(group["fold"].nunique())}
        for metric in metric_cols:
            if metric in group.columns:
                scale = 100.0 if metric in ["WA", "UAR", "Macro_F1", "Weighted_F1"] else 1.0
                row[metric] = f"{group[metric].mean() * scale:.2f} ± {group[metric].std(ddof=0) * scale:.2f}"
        rows.append(row)
    return pd.DataFrame(rows)

if "results_df" in globals() and len(results_df) > 0:
    metric_cols = ["WA", "UAR", "Macro_F1", "Weighted_F1", "CCC_valence", "CCC_arousal", "CCC_dominance", "CCC_mean", "MAE_mean", "RMSE_mean"]
    paper_style = _format_mean_std(results_df, metric_cols)
    paper_style_path = REPORT_DIR / "paper_style_results.csv"
    paper_style.to_csv(paper_style_path, index=False, encoding="utf-8-sig")
    display(paper_style)

    reference_rows = [
        {
            "Model": "Wav2Vec2 P-TAPT (Chen & Rudnicky, 2021)",
            "Modality": "audio",
            "Split": "IEMOCAP session-level",
            "WA/WAR": "",
            "UAR/UA": "74.30",
            "CCC V": "",
            "CCC A": "",
            "CCC D": "",
            "Note": "Pretrained speech fine-tuning reference for 03A",
        },
        {
            "Model": "Ispas et al. 2024 multi-task multi-modal",
            "Modality": "audio + transcript",
            "Split": "10-fold speaker",
            "WA/WAR": "74.69",
            "UAR/UA": "74.68",
            "CCC V": "0.738",
            "CCC A": "0.685",
            "CCC D": "",
            "Note": "Reference for multi-task + bridge/cross-attention",
        },
        {
            "Model": "Proposed notebook branch",
            "Modality": config.get("notebook", "current branch") if "config" in globals() else "current branch",
            "Split": "5-fold + 10-fold",
            "WA/WAR": "; ".join([f"{r['Protocol']}: {r.get('WA', '')}" for _, r in paper_style.iterrows()]),
            "UAR/UA": "; ".join([f"{r['Protocol']}: {r.get('UAR', '')}" for _, r in paper_style.iterrows()]),
            "CCC V": "; ".join([f"{r['Protocol']}: {r.get('CCC_valence', '')}" for _, r in paper_style.iterrows()]),
            "CCC A": "; ".join([f"{r['Protocol']}: {r.get('CCC_arousal', '')}" for _, r in paper_style.iterrows()]),
            "CCC D": "; ".join([f"{r['Protocol']}: {r.get('CCC_dominance', '')}" for _, r in paper_style.iterrows()]),
            "Note": "Direct result from this notebook; compare carefully because modality/split can differ from papers",
        },
    ]
    comparison_df = pd.DataFrame(reference_rows)
    comparison_path = REPORT_DIR / "paper_reference_comparison.csv"
    comparison_df.to_csv(comparison_path, index=False, encoding="utf-8-sig")
    display(comparison_df)

    if "plt" in globals() and plt is not None:
        plot_metrics = [m for m in ["WA", "UAR", "Macro_F1", "CCC_mean"] if m in results_df.columns]
        melted = results_df.melt(id_vars=["protocol", "fold"], value_vars=plot_metrics, var_name="metric", value_name="value")
        fig, ax = plt.subplots(figsize=(9, 4.5))
        if "sns" in globals() and sns is not None:
            try:
                sns.barplot(data=melted, x="metric", y="value", hue="protocol", errorbar="sd", ax=ax)
            except TypeError:
                sns.barplot(data=melted, x="metric", y="value", hue="protocol", ci="sd", ax=ax)
        else:
            for i, metric in enumerate(plot_metrics):
                vals = [results_df.loc[results_df["protocol"] == p, metric].mean() for p in results_df["protocol"].unique()]
                ax.bar([i + j * 0.25 for j in range(len(vals))], vals, width=0.22)
            ax.set_xticks(range(len(plot_metrics)), plot_metrics)
        ax.set_title("Paper-style metric summary by protocol")
        ax.set_ylim(0, 1.0)
        ax.grid(axis="y", alpha=0.25)
        fig.tight_layout()
        fig_path = FIGURE_DIR / "paper_style_metric_summary.png"
        fig.savefig(fig_path, dpi=160)
        plt.show()
        print("Saved:", fig_path)

        history_files = sorted(REPORT_DIR.glob("*_history.csv"))
        if history_files:
            fig, axes = plt.subplots(1, 3, figsize=(15, 4))
            for hp in history_files:
                h = pd.read_csv(hp)
                label = hp.stem.replace("_history", "")
                if "epoch" in h.columns:
                    if "train_loss" in h.columns:
                        axes[0].plot(h["epoch"], h["train_loss"], alpha=0.45)
                    if "val_UAR" in h.columns:
                        axes[1].plot(h["epoch"], h["val_UAR"], alpha=0.45)
                    if "val_CCC_mean" in h.columns:
                        axes[2].plot(h["epoch"], h["val_CCC_mean"], alpha=0.45)
            axes[0].set_title("Train loss by fold")
            axes[1].set_title("Validation UAR by fold")
            axes[2].set_title("Validation CCC mean by fold")
            for ax in axes:
                ax.set_xlabel("epoch")
                ax.grid(alpha=0.25)
            fig.tight_layout()
            fig_path = FIGURE_DIR / "training_curves_by_fold.png"
            fig.savefig(fig_path, dpi=160)
            plt.show()
            print("Saved:", fig_path)

    manifest_rows = []
    for path in sorted(OUTPUT_DIR.rglob("*")):
        if path.is_file():
            relative = path.relative_to(OUTPUT_DIR)
            manifest_rows.append({
                "relative_path": str(relative).replace("\\", "/"),
                "folder": relative.parts[0] if relative.parts else "",
                "bytes": int(path.stat().st_size),
            })
    manifest_df = pd.DataFrame(manifest_rows)
    manifest_df.to_csv(REPORT_DIR / "output_manifest.csv", index=False, encoding="utf-8-sig")
    display(manifest_df)
else:
    print("Chưa có results_df. Hãy chạy cell train trước khi tạo bảng paper-style.")

In [ ]:
config = {
    "notebook": "03A pretrained raw-audio backbone fine-tuning",
    "run_mode": RUN_MODE,
    "pretrained_model_name": PRETRAINED_MODEL_NAME,
    "sample_rate": SAMPLE_RATE,
    "max_seconds": MAX_SECONDS,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "min_delta": MIN_DELTA,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "num_workers": NUM_WORKERS,
    "lr_backbone": LR_BACKBONE,
    "lr_head": LR_HEAD,
    "min_lr_ratio": MIN_LR_RATIO,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "dropout": DROPOUT,
    "hidden_dim": HIDDEN_DIM,
    "unfreeze_last_n": UNFREEZE_LAST_N,
    "label_smoothing": LABEL_SMOOTHING,
    "use_class_weights": USE_CLASS_WEIGHTS,
    "use_scheduler": USE_SCHEDULER,
    "eval_train_split": EVAL_TRAIN_SPLIT,
    "use_data_parallel": USE_DATA_PARALLEL,
    "n_gpus": N_GPUS,
    "run_protocols": RUN_PROTOCOLS,
    "max_folds": MAX_FOLDS,
    "architecture": "pretrained raw audio backbone + attentive statistics pooling + two-head emotion/VAD",
}
(OUTPUT_DIR / "03a_run_config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
zip_output(OUTPUT_DIR)
